# 全ボード・全chのFADC波形積算図をROOTにまとめる

`RawTreeWriter_sm3/raw` の `fadc_ch`・`fadc_wave` を後段で読みます。
波形の横軸をFADCサンプル、縦軸をADC値、色を出現回数とするTH2Dを、1つのROOTにまとめます。
アノード12ボード・カソード12ボード×4chで96個。データのないchも空ヒストグラムで保存します。

イベントごとのTH2を作らず、C++で積算ビンに直接加算します。
TH2Dを使うため、同じビンが大量に埋まってもTH2Fの整数精度制限を避けられます。
ヒストグラムの保持に約0.4 GiB（512サンプルの場合）とROOT入出力のメモリを使います。
全件は大量の波形を展開するため時間がかかります。まず先頭5万イベントで確認してください。

In [ ]:
import sys, subprocess, json
from pathlib import Path
import ROOT
from IPython.display import Image, display
ROOT.gROOT.SetBatch(True)
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'run_script/run_chain.py').exists())
SCRIPT = REPO / 'scripts/analysis/make_tpc_fadc_hist.py'
PYTHON = sys.executable  # このカーネルと同じ、PyROOTのあるPython

## 1. 入力と出力

入力は `RawTreeWriter_sm3` で `save_fadc=true` として保存したROOTです。飛跡ヒットは不要です。
複数runを合わせる場合は `RAWS` にファイルを追加します。同じ測定条件のものを選んでください。
パスは各自の環境に合わせて変更し、大きなデータはリポジトリの外に置きます。
`MAX_EVENTS` は**全入力を通じた**先頭イベント数（波形のあるイベントだけの数ではありません）。`-1` で全件です。
再実行で上書きする場合だけ `OVERWRITE=True` にします。

In [ ]:
RAWS = [Path('/mnt/nas_eureka/onoda/analysis2/SMILE3-FM_analysis/20261113/tpc_raw_20261113_run13.root')]
OUT = Path('/mnt/nas_eureka/onoda/analysis2/SMILE3-FM_analysis/20261113/fadc_accum_run13_n50000')
MAX_EVENTS = 50000   # 積算する先頭イベント数 [events]。全件は -1
OVERWRITE = False   # 既存の積算ROOTを置換する場合のみTrue
WRITE_PNG = True    # 全24ボードの4ch確認図も保存する
OUTPUT_ROOT = OUT / 'fadc_accum.root'
PNG_DIR = OUT / 'png'
for path in [*RAWS, OUT]:
    if not path.is_absolute(): raise ValueError(f'絶対パスを指定してください: {path}')
for path in RAWS:
    if not path.is_file(): raise FileNotFoundError(path)
print('入力:', *RAWS, sep='\n')
print('出力:', OUTPUT_ROOT)

## 2. 積算して保存

閾値や電荷による選別・ベースライン減算は行いません。rawツリーに保存された全波形が対象です。
`save_threshold>0` で保存したrawでは、間引かれた波形を復元できないため警告を出し、その値を記録します。
カソードは `ReadData` によりすでに `1023−ADC` に反転されているため、再反転しません。

ターミナルでも同じスクリプトを使えます（PyROOTのあるPythonで実行）。

```bash
python3 scripts/analysis/make_tpc_fadc_hist.py /path/to/raw.root \
  -o /path/to/output/fadc_accum.root -n 50000 --png-dir /path/to/output/png
# 全件: -n -1。複数入力: raw1.root raw2.root を並べる
```

In [ ]:
cmd = [PYTHON, str(SCRIPT), *map(str, RAWS), '-o', str(OUTPUT_ROOT), '-n', str(MAX_EVENTS)]
if OVERWRITE: cmd.append('--overwrite')
if WRITE_PNG: cmd += ['--png-dir', str(PNG_DIR)]
print(' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)

## 3. 出力ROOTの構成と波形数を確認

`FADC_Accum/FADC_Anode6_Ch0` がA6のローカルch0（ASIC単位）です。
名前のボード番号は物理ID（0〜11）、ch番号はボード内0〜3です。
rawツリーの通し番号はアノード `4×board+ch`、カソード `48+4×board+ch`。
各TH2Dは512×1024ビン（横軸のサンプル数は入力の `run_info.n_fadc_clk` に従います）。
1サンプルは20 ns。ヒットがないボードはrawに波形がない場合があり、波形数は処理イベント数と一致するとは限りません。
`metadata_json` には入力ファイル・処理イベント数・保存閾値・各chの波形数を記録します。
各chのビンの合計は `波形数 × サンプル数` と一致します。

In [ ]:
f = ROOT.TFile.Open(str(OUTPUT_ROOT))
if not f or f.IsZombie(): raise RuntimeError(OUTPUT_ROOT)
metadata = json.loads(str(f.Get('metadata_json').GetString()))
print('処理イベント数:', metadata['events'])
print('ヒストグラム数:', f.Get('FADC_Accum').GetListOfKeys().GetEntries())
for side in ['Anode', 'Cathode']:
    for board in range(12):
        counts = [metadata['channels'][f'FADC_{side}{board}_Ch{ch}']['waveforms'] for ch in range(4)]
        print(f'{side}{board:02d}:', counts)
f.Close()

## 4. 任意のボードの4chを表示

`SIDE` と `BOARD` を変えると確認するボードを選べます。
PNGの色は対数（最小1）、空ビンは空白です。ボードごと・chごとの最大値で色が変わるので、
色の濃さだけで異なるchの波形数を比較せず、メタデータの波形数も確認してください。
PNGは入力イベント数で規格化せず、出現回数を表示します。

In [ ]:
SIDE = 'Anode'  # 'Anode' または 'Cathode'
BOARD = 6       # 物理ボード番号 0〜11
if SIDE not in ('Anode','Cathode') or not 0 <= BOARD <= 11: raise ValueError('ボード指定が不正です')
if WRITE_PNG:
    display(Image(filename=str(PNG_DIR / f'FADC_{SIDE}{BOARD}.png')))
else:
    print('ROOTで', f'FADC_Accum/FADC_{SIDE}{BOARD}_Ch0', '〜 Ch3 を開いて確認できます。')